# Ilera-ATLAS · 01b: can the **raw** N-ATLaS find the right Standing Orders reference?

A baseline **before any fine-tuning**. The base model `NCAIR1/N-ATLaS` is loaded straight from
Hugging Face **as released: no quantisation, no fine-tuning** (float16 weights), and tested on
questions built automatically from the raw Standing Orders chunks (`raw/chunks.jsonl` from
`01_extract`). No human review is needed: every chunk already knows its own section number, title and
pages.

| Test | The model sees | It must answer | What it tells us |
|---|---|---|---|
| `closed_book` | a topic only ("which section covers worm infestation?") | the section number | does N-ATLaS already **know** the Standing Orders? |
| `index` | an excerpt **+ the table of contents** of the document | the section number | can it **map content to its reference** when given the index (retrieval) |
| `mcq` | an excerpt + 4 section options | A/B/C/D | the easy floor: can it recognise the right section at all? (chance = 25%) |

Scores per test: `number_ok` (exact section number), `title_ok` (picked a section with the right title:
lenient, because e.g. EYE DISORDERS exists for children, adults and the elderly), and `mcq_ok`.

## Hardware: runs on a free Colab T4
N-ATLaS has 8B parameters = **~16 GB in float16**. The model is loaded straight with
`device_map="auto"`: whatever fits stays on the GPU and the remaining layers spill onto the
CPU/RAM automatically, so a **single 15 GB T4 works out of the box** (generation is slower than
on a card that holds everything, but no setup is needed).

| Where | GPU | Notes |
|---|---|---|
| **Colab free** | one T4 (15 GB) | works: overflow layers run on the CPU |
| **Kaggle** (free) | **GPU T4 x2** = 2 × 15 GB | faster: the model is split across both GPUs |
| Colab Pro | L4 (24 GB) or A100 | faster still |


**Model files:** downloaded once into the Hugging Face cache (~16 GB). On Colab you can keep a copy in
Google Drive (`SAVE_MODEL_TO_DRIVE = True`, needs **≥17 GB free in Drive**) so later sessions skip
the download. Results are always saved to Drive (Colab) or `/kaggle/working` (Kaggle).

**Local-only run:** `01b` only *reads* from the dataset repo, it **uploads nothing**. Results stay
in this notebook (Google Drive on Colab, `/kaggle/working` on Kaggle); the baseline of record is
the measured numbers in section 6 below.

**Setup:** add the secret `HF_TOKEN` (Colab: key icon → Notebook access on; Kaggle: Add-ons → Secrets),
after accepting the N-ATLaS terms on Hugging Face. Then Run all.

In [ ]:
# ---- Config -------------------------------------------------------------
HF_USER = "YOUR_HF_USERNAME"            # <- your Hugging Face username (owner of ilera-data)
DATA_REPO = f"{HF_USER}/ilera-data"
BASE_MODEL = "NCAIR1/N-ATLaS"           # raw base model: float16, no quantisation, no fine-tuning
USE_DRIVE = True                        # Colab: mount Google Drive for results (ignored on Kaggle)
DRIVE_DIR = "/content/drive/MyDrive/ilera"
SAVE_MODEL_TO_DRIVE = False             # Colab: also keep the 16 GB model files in Drive (needs >= 17 GB free)
DOCS = ["CHEW"]                         # documents to test: "CHEW", "JCHEW", "CHO"
TASKS = ["closed_book", "index", "mcq"]
LIMIT = 40                            # items per task: 40 x 3 tasks = 120 total; None = full set (435)
BATCH = 2                             # keep small: index prompts carry the whole TOC (long KV cache -> OOM at 8)
SEED = 7
LOCAL_CHUNKS = None                     # offline testing: path to a local chunks.jsonl

In [ ]:
import os, subprocess, sys
os.environ.pop("HF_HUB_ENABLE_HF_TRANSFER", None)   # deprecated default on some hosts
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"         # faster model download
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "accelerate", "rapidfuzz"],
                   capture_output=True, text=True)
if r.returncode != 0:                               # pip prints harmless dependency warnings; show them only on failure
    print(r.stdout[-3000:], r.stderr[-3000:])
    raise SystemExit("pip install failed")
print("packages installed")

In [ ]:
import os, sys
def get_hf_token():
    try:
        from google.colab import userdata
        return userdata.get("HF_TOKEN")
    except Exception:
        pass
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass
    import glob                                     # API runs: token in a private Kaggle dataset
    for p in glob.glob("/kaggle/input/**/hf_token.txt", recursive=True):
        return open(p).read().strip()
    return os.environ.get("HF_TOKEN")

HF_TOKEN = get_hf_token()
assert HF_USER != "YOUR_HF_USERNAME", "Set HF_USER in the Config cell"
assert HF_TOKEN, "No HF_TOKEN: add the secret HF_TOKEN (Colab: key icon + Notebook access; Kaggle: Add-ons > Secrets)"
os.environ["HF_TOKEN"] = HF_TOKEN

ON_KAGGLE = os.path.isdir("/kaggle")
HOME = "/kaggle/working/ilera" if ON_KAGGLE else "ilera"
ON_DRIVE = False
if USE_DRIVE and not ON_KAGGLE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        HOME, ON_DRIVE = DRIVE_DIR, True
    except Exception as e:
        print("Google Drive not available; using local disk:", repr(e)[:100])
RESULTS_DIR = os.path.join(HOME, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)
MODEL_DIR = None                                    # None = normal Hugging Face cache
if SAVE_MODEL_TO_DRIVE and ON_DRIVE:
    import shutil
    free_gb = shutil.disk_usage("/content/drive").free / 2**30
    if free_gb >= 17:
        MODEL_DIR = os.path.join(DRIVE_DIR, "models", "N-ATLaS")
    else:
        print(f"Only {free_gb:.0f} GB free in Drive (need 17): the model stays in the session cache instead")
print("results ->", RESULTS_DIR, "| model files ->", MODEL_DIR or "Hugging Face cache (this session)")

## 1. Raw dataset → reference questions
Each **chunk** is one part of a document and carries its true reference (section number, title,
pages). For every chunk we build one `index` item and one `mcq` item from an excerpt in the middle of
the chunk, and for every **section** one `closed_book` item. Items are reproducible (`SEED`).

In [ ]:
import json, random, re, collections

def load_chunks():
    if LOCAL_CHUNKS:
        path = LOCAL_CHUNKS
    else:
        from huggingface_hub import hf_hub_download
        path = hf_hub_download(DATA_REPO, "raw/chunks.jsonl", repo_type="dataset", token=HF_TOKEN)
    return [json.loads(l) for l in open(path, encoding="utf-8") if l.strip()]

chunks = [c for c in load_chunks() if c["doc"] in DOCS]
assert chunks, f"no chunks for {DOCS}: run 01_extract first"

def sec_key(s):
    return tuple(int(x) for x in s.split("."))

sections = {}                                   # (doc, section_id) -> title
for c in chunks:
    sections.setdefault((c["doc"], c["section_id"]), " ".join(c["title"].split()))
toc = {d: "\n".join(f"{sid} {t}" for (dd, sid), t in sorted(sections.items(), key=lambda kv: sec_key(kv[0][1]))
                    if dd == d) for d in DOCS}
DOC_NAME = {"CHEW": "Community Health Extension Workers (CHEW)",
            "JCHEW": "Junior Community Health Extension Workers (JCHEW)",
            "CHO": "Community Health Officers (CHO)"}

def excerpt(text, rng, words=70):
    lines = [l for l in text.splitlines() if len(l.split()) >= 3]
    w = " ".join(lines).split()
    if len(w) <= words:
        return " ".join(w)
    start = rng.randint(len(w) // 4, max(len(w) // 4, len(w) - words - len(w) // 8))
    return " ".join(w[start:start + words])

rng = random.Random(SEED)
items = []
if "closed_book" in TASKS:
    # one question per distinct title: e.g. EYE DISORDERS exists for children, adults and the elderly,
    # so any of its section numbers counts as correct
    by_title = collections.defaultdict(list)
    for (doc, sid), title in sorted(sections.items(), key=lambda kv: (kv[0][0], sec_key(kv[0][1]))):
        by_title[(doc, title.lower())].append(sid)
    for (doc, tl), sids in by_title.items():
        sid, title = sids[0], sections[(doc, sids[0])]
        items.append({"task": "closed_book", "id": f"CB-{doc}-{sid}", "doc": doc, "section_id": sid, "title": title,
                      "accept": sids,
                      "prompt": (f"In the Nigerian National Standing Orders 2024 for {DOC_NAME[doc]}, which section "
                                 f"number covers '{title.title()}'? Answer only in the form: Section <" "number>")})
for c in chunks:
    ex = excerpt(c["text"], rng)
    if "index" in TASKS:
        items.append({"task": "index", "id": f"IX-{c['chunk_id']}", "doc": c["doc"], "section_id": c["section_id"],
                      "title": sections[(c["doc"], c["section_id"])],
                      "prompt": (f"Below is the table of contents of the Nigerian National Standing Orders 2024 for "
                                 f"{DOC_NAME[c['doc']]}, followed by an excerpt from the document.\n\n"
                                 f"TABLE OF CONTENTS:\n{toc[c['doc']]}\n\nEXCERPT:\n\"{ex}\"\n\n"
                                 "Which section is this excerpt from? Answer only in the form: Section <" "number> <" "title>")})
    if "mcq" in TASKS:
        true_t = sections[(c["doc"], c["section_id"])]
        others = [(sid, t) for (d, sid), t in sections.items() if d == c["doc"] and t.lower() != true_t.lower()]
        opts = rng.sample(others, 3) + [(c["section_id"], true_t)]
        rng.shuffle(opts)
        letters = "ABCD"
        gold = letters[[o[0] for o in opts].index(c["section_id"])]
        items.append({"task": "mcq", "id": f"MC-{c['chunk_id']}", "doc": c["doc"], "section_id": c["section_id"],
                      "title": true_t, "gold_letter": gold,
                      "prompt": (f"Excerpt from the Nigerian National Standing Orders 2024 for {DOC_NAME[c['doc']]}:\n"
                                 f"\"{ex}\"\n\nWhich section is it from?\n" +
                                 "\n".join(f"{l}. Section {sid} {t}" for l, (sid, t) in zip(letters, opts)) +
                                 "\n\nAnswer with the letter only.")})
if LIMIT:
    by_task = collections.defaultdict(list)
    for it in items:
        by_task[it["task"]].append(it)
    items = [it for t in TASKS for it in by_task[t][:LIMIT]]
    print(f"*** SMOKE TEST: {LIMIT} items per task ***")
print(len(sections), "sections,", len(chunks), "chunks ->", dict(collections.Counter(it["task"] for it in items)))
print("\nexample index prompt (end):\n..." + next(it for it in items if it["task"] != "closed_book")["prompt"][-500:])

## 2. Load the raw N-ATLaS (float16, no quantisation, no fine-tuning)
`device_map="auto"` puts as much of the model as fits on the GPU and spills the rest to the CPU,
so a single 15 GB T4 works. N-ATLaS ships with its end-of-turn token as padding; for batched
generation it is replaced by its dedicated padding token `<|finetune_right_pad_id|>` (the same fix
Unsloth applies in the other notebooks). Questions are formatted with the model's own chat template
plus the current date (`format_text_for_inference`), the same way a plain N-ATLaS inference call is
made, and generated with `temperature=0.1` and `repetition_penalty=1.12`.

In [ ]:
import datetime
import gc, sys, torch, transformers
from packaging import version
from transformers import AutoModelForCausalLM, AutoTokenizer

globals().pop("model", None)                              # safe to re-run: drop any previous copy
sys.last_type = sys.last_value = sys.last_traceback = None
gc.collect()
assert torch.cuda.is_available(), "No GPU: Kaggle -> Accelerator GPU T4 x2; Colab -> Runtime -> Change runtime type"
torch.cuda.empty_cache()
print("GPU:", torch.cuda.get_device_name(0),
      f"({torch.cuda.mem_get_info(0)[0] / 2**30:.1f} GB free; overflow layers spill to CPU)")

# transformers >= 4.56 calls it `dtype`; older versions use `torch_dtype`
DTYPE_KW = "dtype" if version.parse(transformers.__version__) >= version.parse("4.56") else "torch_dtype"
load_kw = {DTYPE_KW: torch.float16, "device_map": "auto", "token": HF_TOKEN, "low_cpu_mem_usage": True}

source = BASE_MODEL
if MODEL_DIR:                                             # optional copy of the raw files in Google Drive
    if not os.path.exists(os.path.join(MODEL_DIR, "config.json")):
        from huggingface_hub import snapshot_download
        print("copying the raw model files to Drive (one time, ~16 GB) ...")
        snapshot_download(BASE_MODEL, local_dir=MODEL_DIR, token=HF_TOKEN,
                          allow_patterns=["*.json", "*.safetensors", "*.model", "*.txt", "*.jinja"])
    source = MODEL_DIR

print(f"loading {source} in float16 ...")
model = AutoModelForCausalLM.from_pretrained(source, **load_kw)
tok = AutoTokenizer.from_pretrained(source, token=HF_TOKEN)
model.eval()
model.generation_config.max_length = None
tok.padding_side = "left"
if tok.pad_token is None or tok.pad_token == tok.eos_token:
    tok.pad_token = "<|finetune_right_pad_id|>" if "<|finetune_right_pad_id|>" in tok.get_vocab() else tok.eos_token
INPUT_DEVICE = model.get_input_embeddings().weight.device

def format_text_for_inference(messages):
    current_date = datetime.datetime.now().strftime("%d %b %Y")
    return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True,
                                    date_string=current_date)

def chat(user_msgs, max_new=32):
    texts = [format_text_for_inference([{"role": "user", "content": u}]) for u in user_msgs]
    enc = tok(texts, return_tensors="pt", padding=True, add_special_tokens=False).to(INPUT_DEVICE)
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=max_new, do_sample=True, temperature=0.1,
                             repetition_penalty=1.12, pad_token_id=tok.pad_token_id)
    return [tok.decode(o[enc["input_ids"].shape[1]:], skip_special_tokens=True).strip() for o in out]

print(chat(["In one sentence: what is a danger sign of pneumonia in a child?"], max_new=40)[0])

## 3. Ask the questions (resumable: answers are saved as they come)

In [ ]:
import time
OUT = os.path.join(RESULTS_DIR, "base_reference_answers_raw_fp16.jsonl")
done = {}
if os.path.exists(OUT):
    for l in open(OUT, encoding="utf-8"):
        r = json.loads(l); done[r["id"]] = r["answer"]
todo = sorted([it for it in items if it["id"] not in done], key=lambda it: len(it["prompt"]))
print(f"{len(items)} items | {len(done)} already answered | {len(todo)} to ask")
MAX_NEW = {"closed_book": 16, "index": 32, "mcq": 6}
t0, nb = time.time(), (len(todo) + BATCH - 1) // BATCH
with open(OUT, "a", encoding="utf-8") as f:
    for b in range(nb):
        batch = todo[b * BATCH:(b + 1) * BATCH]
        try:
            answers = chat([it["prompt"] for it in batch], max_new=max(MAX_NEW[it["task"]] for it in batch))
        except torch.cuda.OutOfMemoryError:               # long index prompt did not fit: retry one by one
            torch.cuda.empty_cache()
            print(f"  OOM in batch {b + 1}: retrying {len(batch)} items individually", flush=True)
            answers = [chat([it["prompt"]], max_new=MAX_NEW[it["task"]])[0] for it in batch]
        for it, a in zip(batch, answers):
            done[it["id"]] = a
            f.write(json.dumps({"id": it["id"], "answer": a}, ensure_ascii=False) + "\n")
        f.flush()
        if (b + 1) % 5 == 0 or b + 1 == nb:
            el = time.time() - t0
            print(f"  batch {b + 1}/{nb}  {el / 60:.1f} min  ETA {el / (b + 1) * (nb - b - 1) / 60:.0f} min", flush=True)

## 4. Score
* `number_ok`: the first section number in the answer equals the true one (e.g. `2.27`).
* `title_ok`: the answer names a section whose title matches the true title (RapidFuzz ≥ 80), or
  gives a number whose section has the same title (children/adult/elderly duplicates count).
* `mcq_ok`: the first letter A–D equals the correct option.
95% CIs by bootstrap over items.

In [ ]:
import numpy as np
from rapidfuzz import fuzz

NUM = re.compile(r"\b(\d{1,2}(?:\.\d{1,2}){1,2})\b")
title_of = {(d, s): t.lower() for (d, s), t in sections.items()}

def score(it, ans):
    s = {"task": it["task"], "doc": it["doc"], "id": it["id"], "answer": ans,
         "gold": f"{it['section_id']} {it['title']}"}
    if it["task"] == "mcq":
        m = re.search(r"\b([ABCD])\b", ans.strip().upper()[:20])
        s["mcq_ok"] = float(bool(m) and m.group(1) == it["gold_letter"])
        return s
    m = NUM.search(ans)
    num = m.group(1) if m else None
    s["number_ok"] = float(num in it.get("accept", [it["section_id"]]))
    text = NUM.sub("", ans).strip(" :-.").lower()
    same_title = num is not None and title_of.get((it["doc"], num)) == it["title"].lower()
    s["title_ok"] = float(same_title or (len(text) > 3 and fuzz.token_set_ratio(text, it["title"].lower()) >= 80))
    return s

assert score({"task": "index", "doc": "CHEW", "id": "x", "section_id": "2.27", "title": "WORM INFESTATION"},
             "Section 2.27 Worm Infestation")["number_ok"] == 1
assert score({"task": "mcq", "doc": "CHEW", "id": "y", "section_id": "2.3", "title": "FEVER", "gold_letter": "C"},
             "C. Section 2.3 Fever")["mcq_ok"] == 1

scored = [score(it, done[it["id"]]) for it in items if it["id"] in done]
boot = np.random.default_rng(0)
def ci(v):
    v = np.array(v, dtype=float)
    b = boot.choice(v, (2000, len(v))).mean(1)
    return v.mean(), np.percentile(b, 2.5), np.percentile(b, 97.5), len(v)

rows = []
for task in TASKS:
    ss = [s for s in scored if s["task"] == task]
    if not ss:
        continue
    for metric in (["mcq_ok"] if task == "mcq" else ["number_ok", "title_ok"]):
        m, lo, hi, n = ci([s[metric] for s in ss])
        rows.append({"task": task, "metric": metric, "accuracy": round(m, 3), "ci_low": round(lo, 3),
                     "ci_high": round(hi, 3), "n": n, "chance": 0.25 if task == "mcq" else round(1 / len(sections), 3)})
print(f"{'task':12} {'metric':10} {'accuracy':>8}  95% CI          n   chance")
for r in rows:
    print(f"{r['task']:12} {r['metric']:10} {r['accuracy']:>8.3f}  [{r['ci_low']:.3f}, {r['ci_high']:.3f}]  {r['n']:4}  {r['chance']}")

import csv
SUMMARY = os.path.join(RESULTS_DIR, "base_reference_summary_raw_fp16.csv")
DETAIL = os.path.join(RESULTS_DIR, "base_reference_scored_raw_fp16.jsonl")
with open(SUMMARY, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)
with open(DETAIL, "w", encoding="utf-8") as f:
    for s in scored:
        f.write(json.dumps(s, ensure_ascii=False) + "\n")
print("\nsaved:", SUMMARY, "|", DETAIL)
for s in [s for s in scored if s["task"] == "index"][:3]:
    print(f"  gold: {s['gold']:45} | model: {s['answer'][:70]}")

## 5. Real questions: what does the raw model actually say?
The scoring above is numbers; here is the raw model actually **answering real questions**,
using the real Ilera system prompt (`config/system_prompt.txt`). On a fresh dataset that file does
not exist yet (only `02_draft` uploads it): the notebook prints a small "not on HF" note and falls
back to the identical built-in contract text, so the run still works.

* **Pass 1** — closed book: clinical questions with no document. Watch for a valid `ACTION:` line
  and any `SOURCE:` line — the raw model has a clear failure mode: it **fabricates citations**
  (invented documents, plausible-looking section numbers) instead of saying it has none.
* **Pass 2** — grounded: the text of one real Standing Orders chunk is handed to the model, then it
  is asked which section that text comes from. The chunk's gold `SOURCE` line is printed next to
  the answer for comparison.

Expected: medically sensible answers, but **absent or fabricated `SOURCE:` section lines** — the
qualitative twin of the measured index baseline (**15% section number / 20% section title**, n=40)
— and the reason `05_train` puts a correct SOURCE line in every training answer.

In [ ]:
# 1. the real product system prompt (the same text 02_draft trains with)
try:
    from huggingface_hub import hf_hub_download
    _sp = hf_hub_download(DATA_REPO, "config/system_prompt.txt", repo_type="dataset", token=HF_TOKEN)
    SYSTEM = open(_sp, encoding="utf-8").read()
except Exception as e:
    print("system prompt not on HF, using the built-in contract:", repr(e)[:80])
    SYSTEM = ("You are Ilera-ATLAS, a primary health care assistant for trained Nigerian community health workers. "
              "Follow the 2024 National Standing Orders. Answer in the same language as the question.\n"
              "Use exactly this format:\n"
              "ACTION: TREAT | REFER | URGENT\n"
              "WHAT TO DO: numbered steps\n"
              "DANGER SIGNS: signs that mean urgent referral\n"
              "SOURCE: document and section")

def ask_real(question, system=None, max_new=400, context=None):
    msgs = ([{"role": "system", "content": system}] if system else [])
    body = f"{context}\n\nQuestion: {question}" if context else question
    msgs.append({"role": "user", "content": body})
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True,
                                   date_string=datetime.datetime.now().strftime("%d %b %Y"))
    enc = tok(text, return_tensors="pt", add_special_tokens=False).to(INPUT_DEVICE)
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=max_new, do_sample=True, temperature=0.1,
                             repetition_penalty=1.12, pad_token_id=tok.pad_token_id)
    ans = tok.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    return ans, bool(re.search(r"ACTION\s*:", ans, re.I)), bool(re.search(r"SOURCE\s*:", ans, re.I))

def show(tag, q, ans, act, src):
    print(f"\n=== {tag} ===")
    print("Q:", q)
    print(ans[:700] + (" ..." if len(ans) > 700 else ""))
    print(f"[ACTION line: {'yes' if act else 'NO'}]  [SOURCE line: {'yes' if src else 'NO'}]")

print("### Pass 1 - closed book: real questions, Ilera system prompt, no document ###")
for q in ["A 2-year-old has had fever for 3 days and is breathing fast. What should I do?",
          "A child with diarrhoea has sunken eyes and no tears. What should I do?",
          "menene ake nufi da gwagwarmaya?"]:
    ans, act, src = ask_real(q, system=SYSTEM)
    show("closed book", q, ans, act, src)

In [ ]:
print("### Pass 2 - grounded: the real document text is given ###")
# pick the pneumonia chunk: match case-insensitively on the title first, then on the text.
# (the old check compared "PNEUMONIA" against an upper-case title that never contains it
# and silently fell back to chunks[0] - a newborn-care section shown with a pneumonia question.)
chunk = next((c for c in chunks if re.search(r"pneumonia", c["title"], re.I)), None) \
    or next((c for c in chunks if re.search(r"pneumonia", c["text"], re.I)), None)
if chunk is None:
    chunk = chunks[0]
    print(f"WARNING: no pneumonia chunk in {sorted({c['doc'] for c in chunks})}; using "
          f"{chunk['chunk_id']} ({chunk['title']}) - the question below may not match this text")
context = f"From the Nigerian National Standing Orders 2024:\n{chunk['text']}"
Q1 = "A 2-year-old has had fever for 3 days and is breathing fast. What should I do?"
ans, act, src = ask_real(Q1, system=SYSTEM, context=context)
show("grounded answer", Q1, ans, act, src)
print("\nGOLD citation for this chunk:", chunk["source"])
follow = "Which section of the Standing Orders is this from? Answer only in the form: Section <number>"
ans2, act2, src2 = ask_real(follow, system=SYSTEM, context=context)
show("grounded follow-up: which section?", follow, ans2, act2, src2)
print("\n(compare the section number above with the GOLD line: the raw model has never seen this document)")

## 6. What the baseline means for fine-tuning

**Baseline of record** (Kaggle, 2 x T4, n=40 per task): closed_book `number_ok` **2.5%** (1/40),
index `number_ok` **15%** (6/40) / `title_ok` **20%** (8/40), mcq **77.5%** (31/40; chance 25%).
Confidence intervals are wide at n=40 (index 5-27.5%, mcq 65-90%), so re-runs will move the
numbers within that band.

Rules of thumb (the printout below applies them to your numbers):

* **closed_book `number_ok` near 0** (measured 2.5%): N-ATLaS has never seen the 2024 Standing
  Orders, so a `SOURCE:` line can only be correct if the model **learns** the references
  (fine-tuning with the SOURCE line in every answer) or is **given** the text at answer time
  (retrieval/RAG). Section 5 shows the failure mode concretely: the model **fabricates citations**
  — invented documents and section numbers — rather than admitting it has no source.
* **index `number_ok` low (measured 15%) but `mcq_ok` clearly above 25% (measured 77.5%)**: the
  model understands the content but cannot map it to section numbers on its own: fine-tuning on
  Q&A whose answers end with the exact SOURCE helps; adding retrieval of the right chunk (BM25) at
  answer time helps most.
* **mcq near 25%**: the model barely relates excerpts to topics: fine-tuning on Standing Orders
  content is essential, and SOURCE accuracy should be judged on the fine-tuned model with retrieval.

In [ ]:
acc = {(r["task"], r["metric"]): r["accuracy"] for r in rows}
cb, ix, mc = acc.get(("closed_book", "number_ok")), acc.get(("index", "number_ok")), acc.get(("mcq", "mcq_ok"))
print("Baseline (raw N-ATLaS, no fine-tuning):")
if cb is not None:
    print(f"* knows section numbers by heart: {cb:.0%} ->",
          "as expected, it does not know the 2024 Standing Orders" if cb < 0.1 else "some prior knowledge")
if ix is not None:
    print(f"* finds the section from an excerpt + table of contents: {ix:.0%} ->",
          "poor: references must be taught (SOURCE in every training answer) and/or retrieved at answer time"
          if ix < 0.5 else "usable: retrieval + this model may already cite well")
if mc is not None:
    print(f"* picks the right section among 4: {mc:.0%} (chance 25%) ->",
          "understands content well" if mc >= 0.6 else "weak content-to-topic mapping: fine-tuning on the content is essential")
print("\nDone when: these three verdicts match the baseline of record above (closed_book ~2.5%, "
      "index 15%/20%, mcq 77.5%) within the n=40 confidence band.")
print("Next: 02_draft (uploads config/system_prompt.txt) -> 03b_build MODE=\"bench\" -> "
      "03b_build MODE=\"train\" -> 04_eval (re-run: fp16 shifts every score) -> 05_train -> 06_export.")